# Age and Gender Distortion

In a [recent paper in Nature](https://www.nature.com/articles/s41586-025-09581-z), Douglas Guilbeault, Solène Delecourt & Bhargav Srinivasa Desikan investigated the effects of age distortion on genders.

In this assignent, you will work through some of their results yoursefl. You will use their data, available at <https://github.com/drguilbe/distortion_age_gender_online/>.

## Setup

**Python.** Use Python 3.10+. Create a virtual environment and install dependencies so anyone can rerun the notebook:

```bash
python3 -m venv .venv
source .venv/bin/activate 
pip install -r requirements.txt
```

**Packages** (see `requirements.txt`): `pandas`, `numpy`, `matplotlib`, `seaborn`, `statsmodels`, `scipy`, `pingouin`, `scipy.stats`, `plotly`, `jupyter`, `ipykernel`.


## Part 1 - Correlation between age and gender (GPT-2 Large)

We load `GPT2-large-dimensions.csv`. The file includes **three extraction methods** for each construct, following the paper’s robustness checks:

| Construct | Columns (raw scores, not min–max normalized) |
|-----------|-----------------------------------------------|
| Age | `age.main`, `age.ext`, `age.red` |
| Gender | `gender.main`, `gender.ext`, `gender.red` |

The **primary** Pearson correlation matches the replication R script (`fig2_GPT2-large_analyses.R`): **`age.main`** vs **`gender.main`**. Companion columns `*_norm.*` are min–max normalized versions used elsewhere, we use the raw `age.*` / `gender.*` triples for the main correlation and for the robustness heatmaps.

On the **heatmaps**, the main method is labeled **`age_score`** / **`gender_score`** (same values as `age.main` / `gender.main`). Axes follow **`red`, `ext`, `score`** order to match the assignment figures. The **color bars differ**: the age heatmap uses **0.60–1.00**; the gender heatmap uses **0.75–1.00** (both with 0.05 tick spacing within that range).

The summary table uses **Pingouin**’s `corr` (Pearson *r*, 95% CI, *p*, Bayes factor BF10, post-hoc power) so the printed output aligns with the assignment example.

In [3]:
import numpy as np
from pathlib import Path

from matplotlib.cm import ScalarMappable
from matplotlib.colors import LinearSegmentedColormap, Normalize
from mpl_toolkits.axes_grid1 import make_axes_locatable


import matplotlib.pyplot as plt
import pandas as pd
import pingouin as pg
import re
import seaborn as sns


BASE = Path.cwd()
DATA_PATH = BASE / "GPT2-large-dimensions.csv"

df = pd.read_csv(DATA_PATH)

# Heatmap palette: low *r* base -> light coral -> medium red -> darkest red
_CORR_CMAP_COLORS = [
    "#E48568",
    "#F08070",
    "#E86F5A",
    "#E05B4A",
    "#D84A3A",
    "#D1002F",
    "#C8002E",
]


def correlation_colormap() -> LinearSegmentedColormap:
    return LinearSegmentedColormap.from_list("corr_red", _CORR_CMAP_COLORS, N=256)


# Color scale and legend ticks (age vs gender use different ranges, like the assignment figures)
_COLORBAR_AGE_VMIN, _COLORBAR_AGE_VMAX = 0.6, 1.0
_COLORBAR_AGE_TICKS = np.linspace(_COLORBAR_AGE_VMIN, _COLORBAR_AGE_VMAX, 9)

_COLORBAR_GENDER_VMIN, _COLORBAR_GENDER_VMAX = 0.75, 1.0
_COLORBAR_GENDER_TICKS = np.linspace(_COLORBAR_GENDER_VMIN, _COLORBAR_GENDER_VMAX, 6)


def pairwise_corr_matrix(
    frame: pd.DataFrame,
    *,
    cols: list[str],
    labels: list[str],
) -> pd.DataFrame:
    sub = frame[cols].dropna()
    r = sub.corr(method="pearson").reindex(index=cols, columns=cols)
    r.index = labels
    r.columns = labels
    return r


def plot_correlation_heatmap(
    corr: pd.DataFrame,
    *,
    out_path: Path,
    vmin: float,
    vmax: float,
    cbar_ticks: np.ndarray,
    figsize: tuple[float, float] = (4.5, 3.8),
) -> None:
    """Custom palette; pass ``vmin`` / ``vmax`` / ``cbar_ticks`` per plot (age vs gender differ)."""
    vmin_val, vmax_val = vmin, vmax
    cmap = correlation_colormap()
    norm = Normalize(vmin=vmin_val, vmax=vmax_val)

    fig, ax = plt.subplots(figsize=figsize)
    # Colorbar height matches heatmap; explicit colorbar so scale matches vmin/vmax (gender 0.75–1.00 vs age 0.60–1.00)
    divider = make_axes_locatable(ax)
    cax = divider.append_axes("right", size="4.5%", pad=0.12)
    sns.heatmap(
        corr,
        annot=False,
        vmin=vmin_val,
        vmax=vmax_val,
        cmap=cmap,
        square=True,
        linewidths=0.5,
        linecolor="white",
        ax=ax,
        cbar=False,
    )
    sm = ScalarMappable(norm=norm, cmap=cmap)
    sm.set_array([])
    fig.colorbar(sm, cax=cax, ticks=cbar_ticks, format="%.2f")
    ax.set_title("Correlation Heatmap", fontsize=12)

    # White labels on all cells
    for i in range(corr.shape[0]):
        for j in range(corr.shape[1]):
            val = float(corr.iloc[i, j])
            ax.text(
                j + 0.5,
                i + 0.5,
                f"{val:.2f}",
                ha="center",
                va="center",
                color="#FFFFFF",
                fontsize=10,
            )

    plt.tight_layout()
    fig.savefig(out_path, format="svg", bbox_inches="tight")
    plt.close(fig)


# Axis order and *_score labels match the assignment figures
age_r = pairwise_corr_matrix(
    df,
    cols=["age.red", "age.ext", "age.main"],
    labels=["age_red", "age_ext", "age_score"],
)
gender_r = pairwise_corr_matrix(
    df,
    cols=["gender.red", "gender.ext", "gender.main"],
    labels=["gender_red", "gender_ext", "gender_score"],
)

plot_correlation_heatmap(
    age_r,
    out_path=BASE / "correlation_heatmap_age.svg",
    vmin=_COLORBAR_AGE_VMIN,
    vmax=_COLORBAR_AGE_VMAX,
    cbar_ticks=_COLORBAR_AGE_TICKS,
)
plot_correlation_heatmap(
    gender_r,
    out_path=BASE / "correlation_heatmap_gender.svg",
    vmin=_COLORBAR_GENDER_VMIN,
    vmax=_COLORBAR_GENDER_VMAX,
    cbar_ticks=_COLORBAR_GENDER_TICKS,
)


<img src="correlation_heatmap_age.svg" width="300"/><img src="correlation_heatmap_gender.svg" width="300"/>